<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/seq2seq/seq2seq_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

!pip install sentencepiece
import sentencepiece as spm

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


device: cpu


In [3]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in range(len(enc)):
  enc[i] = enc[i].lower()
  dec[i] = dec[i].lower()

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

how can you eat that?                              | comment peux-tu manger ça ?
have you gone mad?                                 | es-tu devenue folle ?
my idea was much better.                           | mon idée était bien meilleure.
tom thinks that the answer is no.                  | tom pense que la réponse est non.
where are my cigarettes?                           | où sont mes cigarettes ?
we saw the children enter the room.                | nous avons vu les enfants rentrer dans la pièce.
i made my dog lie down.                            | j'ai fait s'allonger mon chien.
he holds the rank of colonel.                      | il tient le rang de colonel.
i'll go in first.                                  | je vais entrer en premier.
i had no idea you were so young.                   | je n'avais pas idée que vous étiez si jeunes.


In [4]:
with open("bpe_train.txt", "w", encoding="utf-8") as f:
    for s in enc:
        f.write(str(s).lower() + "\n")
    for s in dec:
        f.write(str(s).lower() + "\n")

spm.SentencePieceTrainer.train(
    input="bpe_train.txt",
    model_prefix="bpe",
    vocab_size=8000,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load("bpe.model")

True

In [5]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 512 # context vector length
max_decoder_seq = 50
batch_size = 128

lr = 0.001
epochs = 10

pad_tok = sp.pad_id()
unk_tok = sp.unk_id()
sos_tok = sp.bos_id()
eos_tok = sp.eos_id()

vocab_len = sp.get_piece_size()
enc_vocab_len = vocab_len
dec_vocab_len = vocab_len

In [6]:
E = []
for sentence in enc:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    E.append(torch.tensor(ids, dtype=torch.long))

D = []
for sentence in dec:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    D.append(torch.tensor(ids, dtype=torch.long))

enc_pad_tok = pad_tok
dec_pad_tok = pad_tok

In [7]:
class Seq2SeqDataset(Dataset):
  def __init__(self, e, d):
    self.e = e
    self.d = d

  def __getitem__(self, index):
     return self.e[index], self.d[index]

  def __len__(self):
    return len(self.e)

""" collate function for batching """
def collate_fn(batch):
  e, d = zip(*batch)

  max_e = max(len(seq) for seq in e)
  max_d = max(len(seq) for seq in d)

  pad_e = torch.zeros(len(e), max_e, dtype=torch.long)
  pad_d = torch.zeros(len(d), max_d, dtype=torch.long)

  for i in range(len(e)):
    pad_e[i] = torch.cat([e[i], torch.tensor([enc_pad_tok] * (max_e - len(e[i])))])

  for i in range(len(d)):
    pad_d[i] = torch.cat([d[i], torch.tensor([dec_pad_tok] * (max_d - len(d[i])))])

  return pad_e, pad_d

In [18]:
class Seq2SeqAttention(nn.Module):
    def __init__(self):
        super(Seq2SeqAttention, self).__init__()

        self.enc_embed = nn.Embedding(enc_vocab_len, embedding_dim, padding_idx=enc_pad_tok)
        self.dec_embed = nn.Embedding(dec_vocab_len, embedding_dim, padding_idx=dec_pad_tok)

        """ encoder-decoder RNN """
        self.num_layers = 1
        self.bidirectional = False

        self.encoder = nn.GRU(embedding_dim, hidden_size,
                              num_layers=self.num_layers,
                              batch_first=True, bidirectional=self.bidirectional)

        self.decoder = nn.GRU(embedding_dim, hidden_size,
                              num_layers=self.num_layers,
                              batch_first=True, bidirectional=self.bidirectional)

        self.fc = nn.Linear(hidden_size * 2, vocab_len)

    def forward(self, e, d=None):
      D = 1 + self.bidirectional
      s = torch.zeros(D * self.num_layers, e.size(0), hidden_size).to(device)
      w = self.enc_embed(e) # word embedding of input sentences

      H, s = self.encoder(w, s) # encoder pass

      y = []
      if d is None:
          token = torch.full((e.size(0), 1), sos_tok, dtype=torch.long, device=device)
          w = self.dec_embed(token) # word embedding of start token

          for _ in range(max_decoder_seq):
            _, s = self.decoder(w, s) # decoder pass

            e_ij = torch.bmm(H, s.squeeze(0).unsqueeze(2))  # alignment model

            mask = (e != enc_pad_tok).unsqueeze(2) # masking pad from softmax
            e_ij = e_ij.masked_fill(~mask, -1e9)

            alpha = torch.softmax(e_ij, dim=1)
            c_i = torch.sum(alpha * H, dim=1) # context vector

            s_c = torch.cat((s.squeeze(0), c_i), dim=1) # adding context vector

            y_i = self.fc(s_c)

            token = torch.argmax(y_i, dim=1).unsqueeze(1)
            y.append(token.squeeze(1))

            w = self.dec_embed(token) # word embedding of predicted token


      else:
        w = self.dec_embed(d)
        for i in range(d.size(1)):
          e_ij = torch.bmm(H, s.squeeze(0).unsqueeze(2))  # alignment model

          mask = (e != enc_pad_tok).unsqueeze(2) # masking pad from softmax
          e_ij = e_ij.masked_fill(~mask, -1e9)

          alpha = torch.softmax(e_ij, dim=1)
          c_i = torch.sum(alpha * H, dim=1).unsqueeze(0) # context vector

          _, s = self.decoder(w[:, i, :].unsqueeze(1), s) # decoder pass
          s_c = torch.cat((s, c_i), dim=2) # adding context vector

          y_i = self.fc(s_c)
          y.append(y_i.squeeze(0))

      return torch.stack(y, dim=1)

In [ ]:
seq2seq_attention = Seq2SeqAttention().to(device)

dataset = Seq2SeqDataset(E[:10000], D[:10000])

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

criterion = nn.CrossEntropyLoss(ignore_index=dec_pad_tok)

optimizer = optim.Adam(
    seq2seq_attention.parameters(),
    lr=lr
)

seq2seq_attention.train()

for epoch in range(epochs):
    total_loss = 0.0

    for e, d in dataloader:
        e = e.to(device)
        d = d.to(device)

        optimizer.zero_grad()

        d_in = d[:, :-1]
        d_out = d[:, 1:]

        out = seq2seq_attention(e, d_in)

        loss = criterion(
            out.reshape(-1, out.size(-1)),
            d_out.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    print(f"Epoch: {epoch+1}/{epochs} | loss: {avg_loss:.4f}")

In [ ]:
seq2seq_attention.eval()

text = """
Yesterday I went to the market with my sister. We bought fresh bread, fruit, and some cheese for dinner.
The weather was cold, but the streets were full of people. After lunch, we visited a small bookstore near the river and
spent almost an hour looking at old novels. I was tired when I got home, but I still watched a movie before going to sleep.
"""

text = sent_tokenize(text)

for test in text:
    seq = sp.encode(test.lower(), out_type=int)
    seq = [sos_tok] + seq + [eos_tok]

    with torch.no_grad():
        seq = torch.tensor(seq, device=device).unsqueeze(0)
        pred = seq2seq_attention(seq)
        pred_ids = []

        for pred_id in pred[0].tolist():
            if pred_id == eos_tok:
                break

            if pred_id not in [sos_tok, dec_pad_tok, pad_tok]:
                pred_ids.append(pred_id)

        print(sp.decode(pred_ids))